In [0]:
from pyspark.sql.functions import current_timestamp, col

RAW_PATH = "/Volumes/retail_project/bronze/raw_files"

files = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

for table_name, filename in files.items():
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{RAW_PATH}/{filename}")
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_file", col("_metadata.file_path"))
    )
    df.write.format("delta").mode("overwrite") \
        .saveAsTable(f"retail_project.bronze.{table_name}")

    print(f"Loaded {table_name}: {df.count()} rows")